In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, size, col, lit, when
from pyspark.sql.types import *

spark = SparkSession.builder\
        .appName("e-commerce_silver_transform")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "2g")\
        .getOrCreate()

In [ ]:
# Initialize project root
project_root = "/home/lpascual/Projects/E-Commerce_DWH/scripts"

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from infrastructure.schemas.glue.datalake_schemas import silver_schemas, gold_schemas 

In [ ]:
# Silver Schema
product_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("product_name", StringType()),
    StructField("brand", StringType()),    
    StructField("product_description", StringType()),
    StructField("primary_category", StringType()),
    StructField("secondary_category", StringType()),
    StructField("tertiary_category", StringType()),
    StructField("is_current", BooleanType()),
    StructField("effective_start_ts", TimestampType()),
    StructField("effective_end_ts", TimestampType()),
    StructField("hash_diff", StringType())
])

event_schema = StructType([
    StructField("event_time", TimestampType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("price", FloatType()),    
    StructField("user_id", LongType()),
    StructField("user_session", StringType())
])

user_schema = StructType([
    StructField("user_id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),    
    StructField("user_name", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType()),
    StructField("is_current", BooleanType()),
    StructField("effective_start_ts", TimestampType()),
    StructField("effective_end_ts", TimestampType()),
    StructField("hash_diff", StringType())    
])

In [ ]:
# Gold Schema
fact_view_schema = StructType([
    StructField("event_id", TimestampType()),
    StructField("user_id", LongType()),    
    StructField("product_id", IntegerType()),
    StructField("date_id", IntegerType()),
    StructField("year", StringType()),
    StructField("month", StringType()),
    StructField("day_of_week", StringType()),
    StructField("primary_category", StringType()),
    StructField("country", StringType()),
    StructField("state", StringType()),
    StructField("city", StringType()),
    StructField("event_type", StringType()),  
    StructField("user_session", StringType()),
    StructField("price_at_event", FloatType())
])

fact_engagement_schema = StructType([
    StructField("event_id", TimestampType()),
    StructField("user_id", LongType()),    
    StructField("product_id", IntegerType()),
    StructField("date_id", IntegerType()),
    StructField("year", StringType()),
    StructField("month", StringType()),
    StructField("day_of_week", StringType()),
    StructField("primary_category", StringType()),
    StructField("country", StringType()),
    StructField("state", StringType()),
    StructField("city", StringType()),
    StructField("event_type", StringType()),  
    StructField("user_session", StringType()),
    StructField("price_at_event", FloatType())
])

dim_product_schema = StructType([
    StructField("id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("name", StringType()),
    StructField("brand", StringType()),    
    StructField("description", StringType()),
    StructField("primary_category", StringType()),
    StructField("secondary_category", StringType()),
    StructField("tertiary_category", StringType()),
    StructField("is_current", BooleanType()),
    StructField("effective_start_ts", TimestampType()),
    StructField("effective_end_ts", TimestampType()),
    StructField("hash_diff", StringType())
])

dim_user_schema = StructType([
    StructField("id", LongType()),
    StructField("user_id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),    
    StructField("user_name", StringType()),
    StructField("sex", StringType()),    
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("is_current", BooleanType()),
    StructField("effective_start_ts", TimestampType()),
    StructField("effective_end_ts", TimestampType()),
    StructField("hash_diff", StringType())    
])

dim_date_schema = StructType([
    StructField("id", IntegerType()),
    StructField("date", DateType()),    
    StructField("day_number_in_month", IntegerType()),
    StructField("day_number_in_year", IntegerType()),
    StructField("day_of_week", StringType()),    
    StructField("calendar_month", StringType()),
    StructField("calendar_month_number", IntegerType()),
    StructField("calendar_year_month", StringType()),
    StructField("calendar_quarter", StringType()),
    StructField("calendar_year", StringType())
])

In [ ]:
import boto3

# -----------------------------
# Spark → Glue type conversion
# -----------------------------
def spark_to_glue_type(spark_type):
    mapping = {
        "IntegerType": "int",
        "LongType": "bigint",
        "StringType": "string",
        "BooleanType": "boolean",
        "TimestampType": "timestamp",
        "FloatType": "float",
        "DoubleType": "double"
    }
    return mapping.get(spark_type, "string")


# -----------------------------
# Convert StructType → Glue Columns
# -----------------------------
def convert_schema(struct_type):
    cols = []
    for field in struct_type.fields:
        glue_type = spark_to_glue_type(type(field.dataType).__name__)
        cols.append({
            "Name": field.name,
            "Type": glue_type
        })
    return cols


# -----------------------------
# Create Glue Table
# -----------------------------
def create_glue_table(
    glue_client,
    database,
    table_name,
    schema,
    s3_location,
    partitions=None
):
    columns = convert_schema(schema)

    partition_keys = []
    if partitions:
        for p in partitions:
            partition_keys.append({"Name": p, "Type": "string"})

    response = glue_client.create_table(
        DatabaseName=database,
        TableInput={
            "Name": table_name,
            "TableType": "EXTERNAL_TABLE",
            "StorageDescriptor": {
                "Columns": columns,
                "Location": s3_location,
                "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
                "SerdeInfo": {
                    "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
                    "Parameters": {"serialization.format": "1"}
                }
            },
            "PartitionKeys": partition_keys,
            "Parameters": {
                "classification": "parquet",
                "EXTERNAL": "TRUE"
            }
        }
    )

    print(f"Created table: {table_name}")
    return response


# -----------------------------
# Glue Client
# -----------------------------
glue = boto3.client("glue", region_name="us-east-1")

# (Paste your schemas here)
# dim_product_schema = ...
# fact_view_schema = ...
# fact_engagement_schema = ...
# dim_user_schema = ...


# -----------------------------
# Create all tables
# -----------------------------
database = "gold_layer"

create_glue_table(
    glue,
    database,
    "dim_product",
    dim_product_schema,
    "s3://your-bucket/gold/dim_product/",
    partitions=["year", "month"]  # optional
)

create_glue_table(
    glue,
    database,
    "fact_view",
    fact_view_schema,
    "s3://your-bucket/gold/fact_view/",
    partitions=["year", "month"]
)

create_glue_table(
    glue,
    database,
    "fact_engagement",
    fact_engagement_schema,
    "s3://your-bucket/gold/fact_engagement/",
    partitions=["year", "month"]
)

create_glue_table(
    glue,
    database,
    "dim_user",
    dim_user_schema,
    "s3://your-bucket/gold/dim_user/",
    partitions=["is_current"]  # optional, or remove
)